<a href="https://colab.research.google.com/github/JoaciQueiroz/Tutor_Idiomas_Interativo/blob/main/Dio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Etapa 1: Captura de Áudio (Frontend)

In [10]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode
import time

# Código JavaScript para capturar áudio
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
    print("🎤 Preparando para gravar...")
    for i in range(3, 0, -1):
        print(f"⏳ {i}...")
        time.sleep(1)
    print(f"▶️ Gravando áudio por {sec} segundos... fale agora!")

    display(Javascript(RECORD))
    js_result = output.eval_js('record(%s)' % (sec * 1000))
    audio = b64decode(js_result.split(',')[1])
    file_name = 'request_audio.wav'
    with open(file_name, 'wb') as f:
        f.write(audio)

    print("✅ Gravação concluída! Processando o áudio...")
    return f'/content/{file_name}'

# Grava o áudio
record_file = record()
display(Audio(record_file, autoplay=False))

🎤 Preparando para gravar...
⏳ 3...
⏳ 2...
⏳ 1...
▶️ Gravando áudio por 5 segundos... fale agora!


<IPython.core.display.Javascript object>

✅ Gravação concluída! Processando o áudio...


# Etapa 2: Reconhecimento de fala com Vosk (gratuito)

In [11]:
!pip install vosk soundfile pydub googletrans==4.0.0-rc1 gtts
!apt-get update && apt-get install -y ffmpeg

from pydub import AudioSegment

input_audio_path = "request_audio.wav"
output_audio_path = "converted_audio.wav"

try:
    audio = AudioSegment.from_file(input_audio_path)
    audio = audio.set_channels(1).set_frame_rate(16000).set_sample_width(2)
    audio.export(output_audio_path, format="wav")
    print("✅ Áudio convertido para WAV compatível:", output_audio_path)
except Exception as e:
    print("❌ Erro ao converter o áudio:", e)

  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
Using cached click-8.1.8-py3-none-any.whl (98 kB)
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
google-adk 1.26.0 requires httpx<1.0.0,>=0.27.0, but you have httpx 0.13.3 which is incompatible.
mcp 1.26.0 requires httpx>=0.27.1, but you have httpx 0.13.3 which is incompatible.
huggingface-hub 1.5.0 requires httpx<1,>=0.23.0, but you have httpx 0.13.3 which is incompatible.
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.

# Etapa 3: Interpretação com ChatGPT

In [12]:
import wave, json
from vosk import Model, KaldiRecognizer

!wget -nc https://alphacephei.com/vosk/models/vosk-model-small-en-us-0.15.zip
!unzip -o vosk-model-small-en-us-0.15.zip

model = Model("vosk-model-small-en-us-0.15")
wf = wave.open("converted_audio.wav", "rb")
rec = KaldiRecognizer(model, wf.getframerate())

results = []
while True:
    data = wf.readframes(4000)
    if len(data) == 0:
        break
    if rec.AcceptWaveform(data):
        results.append(json.loads(rec.Result()))

final_result = json.loads(rec.FinalResult())
results.append(final_result)

user_text = " ".join([res.get("text", "") for res in results]).strip()

if not user_text:
    user_text = ""
print("📝 Transcrição com Vosk:", user_text)

File ‘vosk-model-small-en-us-0.15.zip’ already there; not retrieving.

Archive:  vosk-model-small-en-us-0.15.zip
  inflating: vosk-model-small-en-us-0.15/am/final.mdl  
  inflating: vosk-model-small-en-us-0.15/graph/disambig_tid.int  
  inflating: vosk-model-small-en-us-0.15/graph/HCLr.fst  
  inflating: vosk-model-small-en-us-0.15/graph/Gr.fst  
  inflating: vosk-model-small-en-us-0.15/graph/phones/word_boundary.int  
  inflating: vosk-model-small-en-us-0.15/conf/model.conf  
  inflating: vosk-model-small-en-us-0.15/conf/mfcc.conf  
  inflating: vosk-model-small-en-us-0.15/ivector/splice.conf  
  inflating: vosk-model-small-en-us-0.15/ivector/final.dubm  
  inflating: vosk-model-small-en-us-0.15/ivector/global_cmvn.stats  
  inflating: vosk-model-small-en-us-0.15/ivector/final.ie  
  inflating: vosk-model-small-en-us-0.15/ivector/online_cmvn.conf  
  inflating: vosk-model-small-en-us-0.15/ivector/final.mat  
  inflating: vosk-model-small-en-us-0.15/README  
📝 Transcrição com Vosk: 


# Etapa 4: Conversão em Voz (gTTS)

In [13]:
from googletrans import Translator
translator = Translator()

if user_text.strip():
    translation = translator.translate(user_text, src="pt", dest="en")
    answer = translation.text
else:
    answer = "Blue House."

print("\n💡 Tradução para inglês:")
print(answer)


💡 Tradução para inglês:
Blue House.


# Etapa 5: Reprodução da Resposta (Frontend)

In [14]:
!pip install gtts
!pip install --upgrade click

from gtts import gTTS

tts = gTTS(answer, lang="en")
output_path = "response.mp3"
tts.save(output_path)
print("🔊 Resposta convertida em áudio com gTTS.")

  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
Using cached click-8.3.1-py3-none-any.whl (108 kB)
  Attempting uninstall: click
    Found existing installation: click 8.1.8
    Uninstalling click-8.1.8:
      Successfully uninstalled click-8.1.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.1 which is incompatible.
google-adk 1.26.0 requires httpx<1.0.0,>=0.27.0, but you have httpx 0.13.3 which is incompatible.
mcp 1.26.0 requires httpx>=0.27.1, but you have httpx 0.13.3 which is incompatible.
huggingface-hub 1.5.0 requires httpx<1,>=0.23.0, but you have httpx 0.13.3 which is incompatible.
🔊 Resposta convertida em áudio com gTTS.


# Etapa 6: Reprodução

In [15]:
display(Audio(output_path, autoplay=True))